## Setup

In [ ]:
import sys
import os
import importlib

sys.path.append("../src")

import utils
import metrics

# Reload modules to apply any changes
importlib.reload(utils)
importlib.reload(metrics)

In [ ]:
FILENAME = os.getenv("FILENAME", "UFABC_PLT_combined")
BACKEND = os.getenv("BACKEND", "gemini")
DST = f"../results/{BACKEND}/{FILENAME}"

print(f"{FILENAME=}")
print(f"{BACKEND=}")

df = utils.load(f"../data/embeddings/{BACKEND}/{FILENAME}.csv")

print(f"{df.shape=}")
df.head(3)

## Metrics

In [ ]:
# TODO: understand dataframe "management/updates" here
# grouped = df.groupby(["ID", "Concept"], group_keys=False)

In [ ]:
# TODO: mention what's being computed here
df = (
    df.groupby(["id", "concept"], group_keys=False)
    .apply(
        metrics.compute_distances,
        include_groups=True,
        add_mds=True,
        remove_outliers=False,
        outlier_threshold=1.5,
        min_points=2,
    )
    .reset_index(drop=True)
)

In [ ]:
# TODO: mention what's being computed here
df = (
    df.groupby(["id", "concept"], group_keys=True)
    .apply(
        metrics.compute_dynamics,
        # include_groups=True
    )
    .reset_index(drop=True)
)

In [ ]:
# Keep only original columns + computed metrics (not embeddings)
to_drop = ("prop_embedding", "properties_cum", "embedding", "vel_vector", "acc_vector")
cols = [col for col in df.columns if col not in to_drop]
df = df[cols]

metrics = [
    "entropy",
    "distance_next",
    "distance_centroid_static",
    "vel_magnitude",
    "acc_magnitude",
]
print(f"NaN values: \n\n{df[metrics].isna().mean()}\n")

# Inspect sample of results
df.head(3)

In [ ]:
# Save results
os.makedirs(DST, exist_ok=True)
df[metrics].isna().mean().to_csv(f"{DST}/metrics-nan.csv")
utils.save(df, f"{DST}/metrics.csv")
print(f"{df.shape=}")